In [1]:
import pandas as pd
import os
from collections import defaultdict

def build_single_disease_dataset_from_dirs(label_csv_path, output_csv_path, target_disease):
    """
    构建数据集：
    1. 使用 label_csv_path 中的标签 (negbio)。
    2. 忽略 image_csv_path，改为直接扫描 /course75/MIMIC-Xray/Images/ 下的 PA 和 AP 文件夹。
    3. 根据文件名 (如 50414267_PA_1.jpeg) 匹配 study_id。
    """
    print(f"{'='*10} 开始构建数据集 (基于文件夹扫描) {'='*10}")
    print(f"目标疾病: {target_disease}")
    print(f"标签来源: {label_csv_path}")
    
    # 定义图像文件夹路径
    PA_DIR = '/course75/MIMIC-Xray/Images/PA'
    AP_DIR = '/course75/MIMIC-Xray/Images/AP'
    print(f"扫描图像文件夹:\n  - {PA_DIR}\n  - {AP_DIR}")

    # 1. 读取标签文件 (negbio)
    try:
        # dtype={'study_id': str} 至关重要，确保ID作为字符串处理
        df_labels = pd.read_csv(label_csv_path, dtype={'study_id': str})
    except FileNotFoundError:
        print(f"❌ 错误：找不到标签文件 {label_csv_path}")
        return

    # 检查疾病列是否存在
    if target_disease not in df_labels.columns:
        print(f"❌ 错误：标签文件中没有找到列 '{target_disease}'")
        return

    # 2. 标签筛选 (Label Filtering)
    # 只保留 nebio 文件中标记为 0 或 1 的行
    condition = df_labels[target_disease].isin([0, 1, 0.0, 1.0])
    df_labels_filtered = df_labels[condition].copy()
    
    # 强制转为 int 类型
    df_labels_filtered[target_disease] = df_labels_filtered[target_disease].astype(int)
    
    # 只保留 study_id 和 标签列
    df_labels_clean = df_labels_filtered[['study_id', target_disease]].copy()
    
    print(f"  - 符合标签要求(0/1)的 Study 数量: {len(df_labels_clean)}")

    # 3. 扫描文件夹构建 study_id -> [paths] 的映射 (核心修改)
    # 使用 defaultdict 方便追加路径
    study_id_to_paths = defaultdict(list)
    
    def scan_directory(directory_path):
        if not os.path.exists(directory_path):
            print(f"⚠️ 警告：文件夹不存在 -> {directory_path}")
            return
        
        count = 0
        for fname in os.listdir(directory_path):
            # 假设图片格式为 jpeg, jpg, png 等
            if fname.lower().endswith(('.jpeg', '.jpg', '.png', '.bmp')):
                # 解析文件名：假设文件名格式为 "50414267_PA_1.jpeg"
                # 使用 split('_')[0] 提取 study_id
                try:
                    current_study_id = fname.split('_')[0]
                    full_path = os.path.join(directory_path, fname)
                    study_id_to_paths[current_study_id].append(full_path)
                    count += 1
                except IndexError:
                    continue # 文件名格式不符合预期，跳过
        print(f"  - 在 {os.path.basename(directory_path)} 中找到 {count} 张图片")

    # 执行扫描
    scan_directory(PA_DIR)
    scan_directory(AP_DIR)
    
    print(f"  - 总共找到 {len(study_id_to_paths)} 个唯一的 Study ID 拥有图片")

    # 4. 将路径映射到 DataFrame
    # 这一步会根据 df_labels_clean 中的 study_id 去字典里找对应的路径列表
    # 如果找不到，返回空列表 []
    df_labels_clean['image_paths_list'] = df_labels_clean['study_id'].apply(
        lambda x: study_id_to_paths.get(x, [])
    )

    # 5. 过滤掉没有图片的样本
    # 筛选出 image_paths_list 不为空的行
    final_df = df_labels_clean[df_labels_clean['image_paths_list'].map(len) > 0].copy()

    # 6. 保存
    final_df.to_csv(output_csv_path, index=False)
    
    print(f"✅ 构建完成！")
    print(f"  - 原始标签筛选后数量: {len(df_labels_clean)}")
    print(f"  - 匹配到图像并保留的数量: {len(final_df)}")
    print(f"  - 丢弃(无图像)的数量: {len(df_labels_clean) - len(final_df)}")
    print(f"  - 保存路径: {output_csv_path}")
    
    # 7. 验证展示
    print("\n[预览前5行]:")
    print(final_df.head())

In [6]:
# 配置输入路径
path_to_negbio = 'mimic-cxr-2.0.0-negbio.csv'  # 官方标注文件路径

# 配置目标疾病
target_disease_name = 'Pneumonia'

# 输出文件路径
output_filename = f'./single_class_datasets/dataset_{target_disease_name.replace(" ", "_")}.csv'

# 运行函数 (不再需要传入 image_csv_path)
build_single_disease_dataset_from_dirs(
    path_to_negbio, 
    output_filename, 
    target_disease_name
)

========== 开始构建数据集 (基于文件夹扫描) ==========
目标疾病: Pneumonia
标签来源: mimic-cxr-2.0.0-negbio.csv
扫描图像文件夹:
  - /course75/MIMIC-Xray/Images/PA
  - /course75/MIMIC-Xray/Images/AP
  - 符合标签要求(0/1)的 Study 数量: 40810
  - 在 PA 中找到 95280 张图片
  - 在 AP 中找到 145911 张图片
  - 总共找到 216219 个唯一的 Study ID 拥有图片
✅ 构建完成！
  - 原始标签筛选后数量: 40810
  - 匹配到图像并保留的数量: 37018
  - 丢弃(无图像)的数量: 3792
  - 保存路径: ./single_class_datasets/dataset_Pneumonia.csv

[预览前5行]:
    study_id  Pneumonia                                   image_paths_list
7   50578979          1  [/course75/MIMIC-Xray/Images/AP/50578979_AP_1....
14  51967283          1  [/course75/MIMIC-Xray/Images/AP/51967283_AP_1....
15  54577367          0  [/course75/MIMIC-Xray/Images/AP/54577367_AP_1....
20  58636672          0  [/course75/MIMIC-Xray/Images/PA/58636672_PA_1....
25  53186264          0  [/course75/MIMIC-Xray/Images/PA/53186264_PA_1....
